In [1]:
%pip install numpy pandas matplotlib seaborn scikit-learn scipy statsmodels nltk keras tensorflow torch torchvision bs4 requests lxml openpyxl sqlalchemy


  Using cached statsmodels-0.15.0-cp314-cp314-win_amd64.whl.metadata (9.6 kB)
  Using cached nltk-3.10.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached keras-3.15.1-py3-none-any.whl.metadata (6.4 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for tensorflow


In [12]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3


In [13]:
import os

folder_path = r"C:\Users\ssd m.2\3D Objects\CHURN analysis"

if os.path.exists(folder_path):
    print("Folder found! Here are the files inside it:")
    print(os.listdir(folder_path))
else:
    print("Python can't even find that folder path. Check the folder name spelling!")


Folder found! Here are the files inside it:
['.ipynb_checkpoints', 'churn_banking.db', 'churn_file.ipynb', 'Churn_Modelling.csv']


In [14]:
import pandas as pd
import sqlite3

# 1. The absolute path to the CSV file
csv_path = r"C:\Users\ssd m.2\3D Objects\CHURN analysis\Churn_Modelling.csv"

# 2. Read the CSV file.
df = pd.read_csv(csv_path)

# 3. Connect to (or create) your local SQLite database file
conn = sqlite3.connect("churn_banking.db")

# 4. Push the data into the raw staging table
df.to_sql("raw_churn_data", conn, if_exists="replace", index=False)

# 5. Verify it worked by counting rows
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM raw_churn_data")
row_count = cursor.fetchone()[0]

print(f"Success! Staged {row_count} customer records inside churn_banking.db")

# Close the connection
conn.close()

Success! Staged 10000 customer records inside churn_banking.db


In [15]:
import sqlite3

# 1. opening sqlite db in python 
conn = sqlite3.connect("churn_banking.db")
cursor = conn.cursor()

# 2. Build the Personal Demographics Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS db_customer AS 
SELECT CustomerId, Surname, Geography, Gender, Age FROM raw_churn_data;
""")

# 3. Build the Credit Portfolio Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS db_credit AS 
SELECT CustomerId, CreditScore, Tenure FROM raw_churn_data;
""")

# 4. Build the Account Activity Table
cursor.execute("""
CREATE TABLE IF NOT EXISTS db_account AS 
SELECT CustomerId, Balance, NumOfProducts, HasCrCard, IsActiveMember, EstimatedSalary, Exited FROM raw_churn_data;
""")

# 5. Removes the temporary staging table
cursor.execute("DROP TABLE IF EXISTS raw_churn_data;")

# 6. Commit (save) the structural changes permanently to disk
conn.commit()

# 7. Query the database master catalog to verify the tables exist
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print(f" Phase 1 Success! Current normalized tables: {tables}")

# Close the database channel safely
conn.close()

 Phase 1 Success! Current normalized tables: [('db_customer',), ('db_credit',), ('db_account',)]


In [16]:
import pandas as pd
import sqlite3

# 1. Connect to the normalized database
conn = sqlite3.connect("churn_banking.db")

# 2. Extract high-value, inactive accounts using SQL JOINs
query = """
SELECT 
    c.CustomerId, c.Surname, c.Geography, c.Gender, c.Age,
    cr.CreditScore, cr.Tenure,
    a.Balance, a.NumOfProducts, a.HasCrCard, a.EstimatedSalary, a.Exited
FROM db_account a
JOIN db_customer c ON a.CustomerId = c.CustomerId
JOIN db_credit cr ON a.CustomerId = cr.CustomerId
WHERE a.IsActiveMember = 0 
  AND a.Balance > 50000;
"""

# 3. Load the data into a Pandas DataFrame
df_target = pd.read_sql_query(query, conn)
conn.close()

# 4. Find missing values. 
print(f" Extracted Target Segment: {df_target.shape[0]} customers isolated.")
print("\n--- Missing Value Check ---")
print(df_target.isnull().sum())

print("\n--- Statistical Overview ---")
print(df_target[['Age', 'CreditScore', 'Balance']].describe())

 Extracted Target Segment: 3070 customers isolated.

--- Missing Value Check ---
CustomerId         0
Surname            0
Geography          0
Gender             0
Age                0
CreditScore        0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
EstimatedSalary    0
Exited             0
dtype: int64

--- Statistical Overview ---
               Age  CreditScore        Balance
count  3070.000000  3070.000000    3070.000000
mean     38.492508   648.026710  121393.714205
std       9.154895    98.715593   28646.470179
min      18.000000   350.000000   50911.210000
25%      32.000000   580.000000  101630.235000
50%      38.000000   650.000000  120611.210000
75%      44.000000   717.000000  139856.697500
max      84.000000   850.000000  222267.630000


In [17]:
%pip install --force-reinstall scikit-learn

  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached numpy-2.5.2-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.25.0-py3-none-any.whl (467 kB)
Using cached numpy-2.5.2-cp314-cp314-win_amd64.whl (12.6 MB)
Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl (37.4 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

  Attempting uninstall: threadpoolctl

    Found existing installation: threadpoolctl 3.6.0

    Uninsta

  You can safely remove it manually.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
print("Scikit-Learn imported successfully!")

Scikit-Learn imported successfully!


In [19]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

# 1. Dropping identity columns 
X = df_target.drop(columns=['CustomerId', 'Surname', 'Exited'])
y = df_target['Exited'] # Our target churn label

# 2. One-Hot Encoding on Categorical Features (Geography & Gender)
X_encoded = pd.get_dummies(X, columns=['Geography', 'Gender'], drop_first=True)

# 3.  Min-Max Scaling
scaler = MinMaxScaler()
numerical_cols = ['Age', 'CreditScore', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

# 4. Apply the scaling transformation
X_encoded[numerical_cols] = scaler.fit_transform(X_encoded[numerical_cols])

# 5. Verify the clean mathematical matrix
print(" Phase 2 Success! Raw data transformed into mathematical feature matrix.")
print(f"Final training matrix shape: {X_encoded.shape}")
print("\n--- First 3 Scaled and Encoded Rows ---")




print(X_encoded.head(3))

 Phase 2 Success! Raw data transformed into mathematical feature matrix.
Final training matrix shape: (3070, 10)

--- First 3 Scaled and Encoded Rows ---
        Age  CreditScore  Tenure   Balance  NumOfProducts  HasCrCard  \
0  0.363636        0.304     0.8  0.634640       0.666667          1   
1  0.393939        0.590     0.8  0.366748       0.333333          1   
2  0.166667        0.052     0.4  0.374281       1.000000          1   

   EstimatedSalary  Geography_Germany  Geography_Spain  Gender_Male  
0         0.569544              False            False        False  
1         0.748778              False             True         True  
2         0.596637               True            False        False  


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, roc_auc_score, classification_report

# 1. Splitting the data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

# 2. Initializing the Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)

# 3. Training the algorithm on the historical feature matrix
model.fit(X_train, y_train)

# 4. Extracting continuous Churn Probability Scores for the test set
# Hard Predictions
probabilities = model.predict_proba(X_test)[:, 1]
predictions = model.predict(X_test)

# 5. Calculate and verify our structural performance metrics
precision = precision_score(y_test, predictions)
roc_auc = roc_auc_score(y_test, probabilities)

print(" Phase 3 Success! Predictive model trained and audited.")
print(f" Precision Score: {precision:.2f} (Targeting accuracy for incentive spend)")
print(f" ROC-AUC Score: {roc_auc:.2f} (Model's universal sorting power)")
print("\n--- Detailed Classification Metrics ---")
print(classification_report(y_test, predictions))

 Phase 3 Success! Predictive model trained and audited.
 Precision Score: 0.84 (Targeting accuracy for incentive spend)
 ROC-AUC Score: 0.84 (Model's universal sorting power)

--- Detailed Classification Metrics ---
              precision    recall  f1-score   support

           0       0.83      0.95      0.89       421
           1       0.84      0.56      0.68       193

    accuracy                           0.83       614
   macro avg       0.84      0.76      0.78       614
weighted avg       0.83      0.83      0.82       614



In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Attaching  probabilities to the test data
df_results = X_test.copy()
df_results['Actual_Exited'] = y_test
df_results['Churn_Probability'] = probabilities

# 2. Categorize into strategic Risk Tiers
conditions = [
    (df_results['Churn_Probability'] >= 0.70),
    (df_results['Churn_Probability'] >= 0.40) & (df_results['Churn_Probability'] < 0.70),
    (df_results['Churn_Probability'] < 0.40)
]
choices = ['High Risk', 'Medium Risk', 'Low Risk']
df_results['Risk_Tier'] = np.select(conditions, choices, default='Low Risk')

# 3. Print Business Segmentation Summary
print("✨ Phase 4 Success! Deployment Tiers Generated.")
print("\n--- Risk Tier Distribution ---")
print(df_results['Risk_Tier'].value_counts())

print("\n--- Actual Churn Rate per Tier ---")
print(df_results.groupby('Risk_Tier')['Actual_Exited'].mean().apply(lambda x: f"{x*100:.1f}%"))

# 4. Extract Feature Importances to explain model drivers
importances = model.feature_importances_
feature_names = X_encoded.columns
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\n--- Top Churn Drivers ---")
print(feature_importance_df.head(5))

✨ Phase 4 Success! Deployment Tiers Generated.

--- Risk Tier Distribution ---
Risk_Tier
Low Risk       458
High Risk       95
Medium Risk     61
Name: count, dtype: int64

--- Actual Churn Rate per Tier ---
Risk_Tier
High Risk      89.5%
Low Risk       14.6%
Medium Risk    67.2%
Name: Actual_Exited, dtype: object

--- Top Churn Drivers ---
           Feature  Importance
0              Age    0.479915
4    NumOfProducts    0.159153
3          Balance    0.097351
6  EstimatedSalary    0.072799
1      CreditScore    0.069856
